In [18]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions") #konwencja kropka na poczatku
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [7]:
from pyspark.sql.functions import sum as _sum, min as _min, max as _max

store_summary = (
    df.groupBy("category")
    .agg(
        _sum("amount").alias("sum_amount"),
        _min("amount").alias("sum_amount"),
        _max("amount").alias("sum_amount"),
    )
    .orderBy("category")
)
store_summary.show()

+-----------+------------------+----------+----------+
|   category|        sum_amount|sum_amount|sum_amount|
+-----------+------------------+----------+----------+
|elektronika|1520770.6899999995|       9.0|    9999.0|
|    książki| 851382.0799999991|       5.0|   9107.25|
|     odzież| 849877.5500000007|       5.0|   9696.63|
|    żywność| 789514.4300000003|       5.0|   6916.92|
+-----------+------------------+----------+----------+



In [8]:
from pyspark.sql.functions import sum as _sum, min as _min, max as _max

df.groupBy("category") \
  .agg(
      _sum("amount").alias("sum_amount"),
      _min("amount").alias("min_amount"),
      _max("amount").alias("max_amount")
  ) \
  .orderBy("category") \
  .show()

+-----------+------------------+----------+----------+
|   category|        sum_amount|min_amount|max_amount|
+-----------+------------------+----------+----------+
|elektronika|1520770.6899999995|       9.0|    9999.0|
|    książki| 851382.0799999991|       5.0|   9107.25|
|     odzież| 849877.5500000007|       5.0|   9696.63|
|    żywność| 789514.4300000003|       5.0|   6916.92|
+-----------+------------------+----------+----------+



In [9]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [10]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [20]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [21]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Okna sliding (przesuwne) nachodzą na siebie, podczas gdy okna tumbling (nieprzesuwne) są rozłączne.
# W przypadku tumbling każde zdarzenie należy dokładnie do jednego okna (partycjonowanie czasu bez nakładania).
# Natomiast w przypadku sliding, ponieważ okno ma długość 1h, a przesunięcie wynosi 30 minut,
# każde zdarzenie może należeć do kilku okien (zwykle do dwóch).
# W efekcie liczba wygenerowanych okien (a więc i wierszy po agregacji) jest większa dla sliding.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [25]:
# ZADANIE DOMOWE 
# 1. Najniższa średnia kwota – sklep Gdańsk
from pyspark.sql.functions import window, col, avg

gdn_avg = (
    df
    .filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(avg("amount").alias("avg_amount"))
)

lowest_hour = gdn_avg.orderBy("avg_amount").limit(1)

lowest_hour.show(truncate=False)

+------------------------------------------+-----------------+
|window                                    |avg_amount       |
+------------------------------------------+-----------------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.0118407310706|
+------------------------------------------+-----------------+



In [27]:
# 2. Liczba transakcji per kategoria (09:00–09:30)

from pyspark.sql.functions import count, hour, minute, col

transactions_0930 = (
    df
    .filter(
        (hour(col("timestamp")) == 9) &
        (minute(col("timestamp")) < 30)
    )
    .groupBy("category")
    .agg(count("tx_id").alias("tx_count"))
    .orderBy("category")
)

transactions_0930.show(truncate=False)

+-----------+--------+
|category   |tx_count|
+-----------+--------+
|elektronika|611     |
|książki    |622     |
|odzież     |605     |
|żywność    |567     |
+-----------+--------+



In [28]:
# 3. Okna 15-minutowe – szczyt transakcji
from pyspark.sql.functions import window, count, col

peak_15min = (
    df
    .groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("tx_count"))
    .orderBy(col("tx_count").desc())
)

peak_15min.show(truncate=False)

+------------------------------------------+--------+
|window                                    |tx_count|
+------------------------------------------+--------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234    |
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171    |
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156    |
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139    |
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100    |
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899     |
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858     |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644     |
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582     |
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468     |
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|443     |
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|306     |
+------------------------------------------+--------+



In [29]:
peak_15min.limit(1).show(truncate=False)

+------------------------------------------+--------+
|window                                    |tx_count|
+------------------------------------------+--------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234    |
+------------------------------------------+--------+

